# 3.1 — Tabular data, CSV, and pandas

Move the list of dictionaries from 2.3 into a table, load CSV data, and inspect column meaning and types before calculating.

## Introduction

Use this Notebook to verify the lesson concepts with actual data and code.

## Learning outcomes

- Explain what one row, one column, and one cell represent in a table.
- State CSV loading assumptions and identify the file actually loaded.
- Inspect a DataFrame's shape, columns, inferred types, missingness, and raw categories.
- Add derived columns and save a CSV without an unintended index.

> **Learning route:** Required: 3.1.1–3.1.4 / Supporting: 3.1.5 / Integrated practice: 3.1.6


## 3.1.1 Understand rows, columns, cells, and schema

In this course, one row is one centre-month observation, one column is a variable with consistent meaning, and one cell is one variable value for one observation. Treat the header plus each column's expected type, unit, and missing-value rule as its schema.

In [ ]:
import pandas as pd

records = [
    {"month": "2026-01", "centre_id": "C001", "registered": 32, "completed": 24},
    {"month": "2026-01", "centre_id": "C002", "registered": 27, "completed": 18},
]
records_df = pd.DataFrame(records)
records_df


## 3.1.2 Understand CSV and loading assumptions

A CSV commonly uses its first line as a header, later lines as records, and commas as field delimiters. A value containing a comma or newline needs quoting. Consider encoding, delimiter, empty fields that represent missing data, and identifiers whose leading zero must be retained.

### Relative paths and the file actually loaded

`data/file.csv` is interpreted from the kernel's current working directory, not necessarily the Notebook file's directory. To work from different Python Lab entry points, the helper below checks the current location, its parents, the server work area, and the distributed materials. Always inspect the printed `Working directory` and `Loading` paths.

In [ ]:
from pathlib import Path
import pandas as pd


def find_course_data(filename):
    """Find course data without depending on the Notebook start directory."""
    roots = [
        Path.cwd(),
        *Path.cwd().parents,
        Path.home() / "work",
        Path("/opt/python-lab/course-materials"),
    ]
    checked = []
    for root in roots:
        for candidate in (root / "data" / filename, root / filename):
            candidate = candidate.expanduser()
            if candidate in checked:
                continue
            checked.append(candidate)
            if candidate.is_file():
                return candidate
    locations = "\n".join(f"- {path}" for path in checked)
    raise FileNotFoundError(
        f"Course data file {filename!r} was not found. Checked:\n{locations}"
    )


data_file = find_course_data("learning-centres-practice.csv")
print("Working directory:", Path.cwd())
print("Loading:", data_file.resolve())


### Make CSV loading assumptions explicit

Import `pandas` conventionally as `pd`. `read_csv()` creates a `DataFrame`. Here UTF-8 is explicit, and centre codes and months remain strings rather than quantities. Other files may require a different delimiter or encoding.

In [ ]:
df = pd.read_csv(
    data_file,
    encoding="utf-8",
    dtype={"centre_id": "string", "month": "string"},
)
print(df.head(3))


## 3.1.3 Load and inspect a DataFrame

Use `head()` for representative layout, `shape` for row and column counts, `columns` for exact names, `dtypes` and `info()` for inferred types, and `isna().sum()` for missing counts. If they differ from the expected schema, investigate the input or loading options before calculation.

### Display every row and count raw category values

`head()` is for a first sample. For a human-inspectable source, `to_string(index=False)` displays every row without an ellipsis. Count raw category values with `value_counts(dropna=False, sort=False)` and convert the Series to a named two-column table when needed.

In [ ]:
print(df.to_string(index=False, line_width=200))

district_counts = (
    df["district"]
    .value_counts(dropna=False, sort=False)
    .rename_axis("district")
    .reset_index(name="records")
)
print(district_counts.to_string(index=False, formatters={"district": repr}))


In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Dtypes:")
print(df.dtypes)
print("Missing values:")
print(df.isna().sum())
print("Info:")
df.info()


### One column is a Series; multiple columns form a DataFrame

`df["registered"]` returns a one-dimensional `Series`. `df[["registered"]]` returns a two-dimensional `DataFrame` with one column. Specify column names exactly and use two bracket pairs for a list of selected columns.

In [ ]:
registered_series = df["registered"]
registered_table = df[["registered"]]
print(type(registered_series).__name__, registered_series.shape)
print(type(registered_table).__name__, registered_table.shape)


## 3.1.4 Calculate columns and save a table

Series arithmetic applies across corresponding rows. `assign()` can create a new DataFrame containing a derived column without directly changing the source `df`. Lesson 3.3 examines zero denominators and invalid values in detail.

In [ ]:
report = df.assign(
    completion_rate=df["completed"] / df["registered"] * 100
)
print(report[["month", "centre_name", "registered", "completed", "completion_rate"]].head())


### Distinguish the DataFrame index from an operational identifier

The index shown at the left is a pandas row label, not a replacement for `centre_id`. Use `index=False` when exporting if that extra index column is not part of the data. Check the saved path and header afterward.

In [ ]:
preview_file = Path.cwd() / "monthly-centres-preview.csv"
report.head(5).to_csv(preview_file, index=False, encoding="utf-8")
print("Saved:", preview_file.resolve())
print(preview_file.read_text(encoding="utf-8").splitlines()[0])


## 3.1.5 Diagnose loading problems by cause

For `FileNotFoundError`, inspect the working directory and checked paths. If every field becomes one column, inspect the delimiter. For mojibake or `UnicodeDecodeError`, inspect encoding. If a numeric column becomes text, inspect units, spaces, and invalid values. Successfully loading a file is not proof that it was loaded correctly.

## 3.1.6 Integrated practice: build, save, re-read, and reconcile a table

Create a DataFrame from the centre records in 2.3, save it to CSV, and load it again. Confirm that shape, columns, identifiers, and count totals agree before and after. Add `attendance_rate` and `completion_rate`. Lesson 3.2 then selects rows and columns that answer a question.

In [ ]:
# Write the transfer solution here.


## Summary

- Defined the observation represented by a row and the variable represented by a column.
- Checked the CSV location and loading assumptions before interpreting the DataFrame.
- Inspected the complete table, then calculated and saved a separate result.

## Next

The table is now inspectable. Lesson 3.2 turns an analysis question into displayed columns and reproducible row conditions.

**Estimated learning time:** about 4 hours
